In [8]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import importlib
import create_population.import_population as pop
import simulation.sample_size_calculation as ss

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
importlib.reload(pop)
importlib.reload(ss)

<module 'simulation.sample_size_calculation' from 'c:\\Users\\isabe\\Desktop\\PhD\\Articles\\A1. Artigo MUS standard\\MUS_Article\\testing\\..\\simulation\\sample_size_calculation.py'>

In [10]:
from itertools import product
from math import prod
from scipy.stats import norm
import pandas as pd
from tqdm import tqdm

In [11]:
POPULATION_CONFIGS  = {
    "BV_pop": ["BV_5pct_above_SI", "BV_15pct_above_SI", "BV_30pct_above_SI"],
    "f_target": [0.05, 0.20, 0.50],
    "corr_target": [0.10, 0.25, 0.50],
    "r_target": [0.002, 0.01],
    "cl": [0.80, 0.90, 0.95],
    "file": ["simulated_error_populations.xlsx", "simulated_error_populations.FIRST_BACKUP.xlsx"]
}

In [12]:
rows = []

keys = [
    "file",
    "BV_pop",
    "f_target",
    "corr_target",
    "r_target",
    "cl",
]

total = prod(len(POPULATION_CONFIGS[k]) for k in keys)

for file, hv, freq, corr, er, cl in tqdm(
    product(
        POPULATION_CONFIGS["file"],
        POPULATION_CONFIGS["BV_pop"],
        POPULATION_CONFIGS["f_target"],
        POPULATION_CONFIGS["corr_target"],
        POPULATION_CONFIGS["r_target"],
        POPULATION_CONFIGS["cl"],
    ),
    total=total,
    desc="Calculating sample sizes",
):
    config = {
        "BV_pop": hv,
        "f_target": freq,
        "corr_target": corr,
        "r_target": er,
        "file": file
    }

    population = pop.import_population(**config)
    BV = population["BV"].sum()
    sample_size = ss.sample_size_HH(
        BV=BV,
        z_score=norm.ppf(cl),
        TE=0.02*BV,
        std=(population["E"] / population["BV"]).std(ddof=1),
        AE=population["E"].sum()
    )

    rows.append(
        config | {
            "cl": cl,
            "sample_size": sample_size
        }
    )

sample_size_df = pd.DataFrame(rows)

Calculating sample sizes: 100%|██████████| 324/324 [00:46<00:00,  7.01it/s]


In [13]:
sample_size_df

,BV_pop,f_target,corr_target,r_target,file,cl,sample_size
0,BV_5pct_above_SI,0.05,0.1,0.002,simulated_error_populations.xlsx,0.80,1
1,BV_5pct_above_SI,0.05,0.1,0.002,simulated_error_populations.xlsx,0.90,2
2,BV_5pct_above_SI,0.05,0.1,0.002,simulated_error_populations.xlsx,0.95,3
3,BV_5pct_above_SI,0.05,0.1,0.010,simulated_error_populations.xlsx,0.80,43
4,BV_5pct_above_SI,0.05,0.1,0.010,simulated_error_populations.xlsx,0.90,98
...,...,...,...,...,...,...,...
319,BV_30pct_above_SI,0.50,0.5,0.002,simulated_error_populations.FIRST_BACKUP.xlsx,0.90,16
320,BV_30pct_above_SI,0.50,0.5,0.002,simulated_error_populations.FIRST_BACKUP.xlsx,0.95,26
321,BV_30pct_above_SI,0.50,0.5,0.010,simulated_error_populations.FIRST_BACKUP.xlsx,0.80,194
322,BV_30pct_above_SI,0.50,0.5,0.010,simulated_error_populations.FIRST_BACKUP.xlsx,0.90,450


In [14]:
sample_size_df.to_excel('../results/sample_size_analysis.xlsx')